# 01 — Data audit and cleaning decisions


This notebook establishes a traceable evidence trail from the immutable UK-Air export to the modelling table. It does not silently delete or fill values. The cleaning policy is fixed before model evaluation: negative readings are invalid, duplicate timestamps are resolved deterministically, the hourly grid is completed, bounded gaps of at most three hours are linearly interpolated, and every longer gap remains missing.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
RAW=ROOT/"data/raw/London_Marylebone_PM25.csv"
PROCESSED=ROOT/"data/processed"
FIGURES=ROOT/"figures"
RESULTS=ROOT/"results"
import json
import pandas as pd
from pm25forecast.io import read_uk_air_csv
from pm25forecast.quality import annotate_measurements, raw_audit
from pm25forecast.preprocessing import preprocess

In [ ]:
raw=read_uk_air_csv(RAW)
annotated=annotate_measurements(raw)
print(raw_audit(annotated))
annotated[["datetime","pm25_raw","pm25_observed","quality_issue","instrument_name"]].head()

## Execute the official preprocessing pipeline

The generated files are versioned by purpose: annotated raw rows, complete cleaned grid, model-ready features, annual coverage, gap inventory, and a JSON audit.

In [ ]:
model_ready,audit,gaps,coverage=preprocess(RAW,PROCESSED,short_gap_limit=3)
print(json.dumps(audit,indent=2))
coverage.style.format({"coverage":"{:.2%}"})

In [ ]:
gaps.groupby("gap_class").agg(gaps=("gap_id","count"),hours=("hours","sum"),largest=("hours","max")).sort_values("largest",ascending=False)

In [ ]:
gaps.head(15)

## Checks required before modelling

The assertions below fail loudly if the dataset no longer matches the documented quality policy.

In [ ]:
assert audit["negative_values"]==238
assert audit["missing_or_non_numeric"]==8087
assert audit["largest_gap_hours"]==1656
assert model_ready.loc[model_ready.long_gap_preserved,"pm25_clean"].isna().all()
assert not model_ready.loc[model_ready.was_interpolated,"pm25_clean"].isna().any()
print("Cleaning invariants passed.")